In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Generate a simulated electricity dataset
rng = np.random.default_rng(42)
n = 500

df = pd.DataFrame({
    "operating_hours": rng.uniform(1, 12, n),
    "device_count": rng.integers(1, 21, n),
    "avg_temp_c": rng.uniform(18, 35, n)
})

noise = rng.normal(0, 8.0, n)

df["energy_kwh"] = (
    1.8 * df["operating_hours"]
    + 0.35 * df["device_count"]
    + 0.08 * df["avg_temp_c"]
    + noise
)

print(df.head())
print("\nDataset shape:", df.shape)

# 2. Define features and target
X = df[["operating_hours", "device_count", "avg_temp_c"]]
y = df["energy_kwh"]

# 3. Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 4. Build and train Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

# 5. Predict
y_pred = model.predict(X_test)

# 6. Build a simple baseline
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

# 7. Evaluate both models
def evaluate_model(name, y_true, y_hat):
    mae = mean_absolute_error(y_true, y_hat)
    rmse = np.sqrt(mean_squared_error(y_true, y_hat))
    r2 = r2_score(y_true, y_hat)

    print(f"\n{name}")
    print(f"MAE:  {mae:.3f} kWh")
    print(f"RMSE: {rmse:.3f} kWh")
    print(f"R2:   {r2:.3f}")

evaluate_model("Baseline (predict mean)", y_test, baseline_pred)
evaluate_model("Linear Regression", y_test, y_pred)

# 8. Predict for a new example
new_day = pd.DataFrame({
    "operating_hours": [8],
    "device_count": [10],
    "avg_temp_c": [30]
})

prediction = model.predict(new_day)[0]

print(f"\nPredicted energy consumption: {prediction:.2f} kWh")

   operating_hours  device_count  avg_temp_c  energy_kwh
0         9.513517            17   22.425925    3.654737
1         5.827663            15   23.482337    6.594306
2        10.444577            12   22.122209   10.285066
3         8.671048            18   26.157678    6.002231
4         2.035951             3   29.615392   -2.478929

Dataset shape: (500, 4)

Baseline (predict mean)
MAE:  8.280 kWh
RMSE: 10.398 kWh
R2:   -0.000

Linear Regression
MAE:  7.715 kWh
RMSE: 9.410 kWh
R2:   0.181

Predicted energy consumption: 20.02 kWh
